#### **Player Data**
This file uses multiple sources to create a file with player data, such as id, name, jersey number, physical characteristics and previous season numbers

#### **Imports**

In [ ]:
import pandas as pd
import json
import re
import unicodedata
pd.set_option('display.max_rows', None)

#### **Data Cleaning**

In [ ]:
# Cleaning the height/weight file

with open('../../data/player/original/height_weight.json', encoding='latin-1') as f:
    data = json.load(f)

players = data['players']

height_weight = pd.DataFrame([{
    'name': p['name'],
    'pos': p['pos'],
    'hgt': p['hgt'],
    'weight': p['weight'],
    'tid': p['tid'],
    'college': p.get('college', None),
    'birth_year': p['born']['year'],
    'birth_loc': p['born']['loc']
} for p in players])

# Manually fix some names that are different in the player name id dataset
name_corrections = {
    'Louis Williams': 'lou williams',
    'Otto Porter': 'otto porter jr',
    'Kelly Oubre': 'kelly oubre jr',
    'Luc Richard Mbah a Moute': 'luc mbah a moute',
    "Johnny O'Bryant": 'johnny obryant iii',
    'James Ennis': 'james ennis iii',
    'Joseph Young': 'joe young',
    'James McAdoo': 'james michael mcadoo',
    'Louis Amundson': 'lou amundson',
    'Walter Tavares': 'edy tavares',
}
height_weight['name'] = height_weight['name'].replace(name_corrections)

# normalize player names - all lowercase, remove accents, and remove punctuation
height_weight['name'] = height_weight['name'].str.lower()
height_weight['name'] = height_weight['name'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
height_weight['name'] = height_weight['name'].str.replace(r'[^\w\s]', '', regex=True)


height_weight = height_weight[['name', 'hgt', 'weight']]
height_weight.head()

In [ ]:
# Cleaning the jersey numbers file

with open('../../data/player/original/jersey_numbers.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

name_corrections = {
    "enes freedom": "enes kanter",
    "omer ask": "omer asik",
    "johnny obryant": "johnny obryant iii",
    "tibor plei": "tibor pleiss"
}

data = []
current_jersey = None

for line in lines:
    # remove asterisk and trim
    line = line.replace('*', '').strip()
    if not line:
        continue

    # jersey-number line
    if line.isdigit():
        current_jersey = line
        continue

    # player/team line: split only on last space
    parts = line.rsplit(' ', 1)
    if len(parts) != 2:
        continue

    raw_name, raw_teams = parts

    # normalize only player name
    name = raw_name.lower()
    name = unicodedata.normalize('NFKD', name).encode('ascii', errors='ignore').decode('utf-8')
    name = re.sub(r'[^\w\s]', '', name)
    name = name_corrections.get(name, name)

    team_list = [t.strip().upper() for t in raw_teams.split('-')]
    team_list = ['CHA' if t == 'CHO' else t for t in team_list]

    for team in team_list:
        data.append({
            'name': name,
            'team': team,
            'jersey_number': current_jersey
        })

jersey_numbers = pd.DataFrame(data)
jersey_numbers.head()

In [ ]:
# Cleaning the player name id file

player_name_id = pd.read_csv('../../data/player/original/player_name_id.csv', encoding='utf-8')
player_name_id = player_name_id[['NBAName', 'NBAID']]

# Remove nan
player_name_id = player_name_id.dropna()

# Convert NBAID to int
player_name_id['NBAID'] = player_name_id['NBAID'].astype(int)

# normalize player names - all lowercase, remove accents, and remove punctuation
player_name_id['NBAName'] = player_name_id['NBAName'].str.lower()
player_name_id['NBAName'] = player_name_id['NBAName'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
player_name_id['NBAName'] = player_name_id['NBAName'].str.replace(r'[^\w\s]', '', regex=True)

# Remove duplicates (only happens one time)
player_name_id = player_name_id.drop_duplicates(subset=['NBAID'], keep='first')

player_name_id.head()

In [ ]:
# Cleaning the player teams file

player_teams = pd.read_csv('../../data/player/original/player_teams.csv', encoding='utf-8')
player_teams = player_teams[['Player', 'Team', 'Pos']]
# Change 'CHO' to 'CHA' for Charlotte Hornets
player_teams['Team'] = player_teams['Team'].replace('CHO', 'CHA')

# Remove '2TM' and '3TM' rows
player_teams = player_teams[~player_teams['Team'].isin(['2TM', '3TM'])]

# normalize player names - all lowercase, remove accents, and remove punctuation
player_teams['Player'] = player_teams['Player'].str.lower()
player_teams['Player'] = player_teams['Player'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
player_teams['Player'] = player_teams['Player'].str.replace(r'[^\w\s]', '', regex=True)

# Manually changing some player names to match the format of the other datasets
name_corrections = {
    "enes freedom": "enes kanter",
    "omer ask": "omer asik",
    "johnny obryant": "johnny obryant iii",
    "tibor plei": "tibor pleiss"
}

player_teams['Player'] = player_teams['Player'].replace(name_corrections)
                                                            
player_teams.head()

In [ ]:
# Cleaning the team id file

team_id = pd.read_csv('../../data/player/original/team_id.csv', encoding='latin-1')
team_id = team_id[team_id['Season'] == 2016]
team_id = team_id[['BBRef_Team_Abbreviation', 'NBA_Current_Link_ID']]
team_id.head()

In [ ]:
# Cleaning the stats file

players_stats_2014_15 = pd.read_csv('../../data/player/original/2014_15_player_stats.csv', encoding='utf-8')
players_stats_2014_15 = players_stats_2014_15[['Player', 'G', 'ORB', 'DRB', 'TRB']]

# normalize player names - all lowercase, remove accents, and remove punctuation
players_stats_2014_15['Player'] = players_stats_2014_15['Player'].str.lower()
players_stats_2014_15['Player'] = players_stats_2014_15['Player'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
players_stats_2014_15['Player'] = players_stats_2014_15['Player'].str.replace(r'[^\w\s]', '', regex=True)

# some players appear more than one time because they were traded during the season
# maintain only the row with the most games played
players_stats_2014_15 = players_stats_2014_15.sort_values('G', ascending=False).drop_duplicates(subset=['Player'], keep='first')

players_stats_2014_15.head()

In [ ]:
# Some players have the same name
# The best solution is to check for players with the same name and filter out the ones that did not play in the 2016 season.
# The list of players left is short and can be checked for them manually.

# Check for players with the same name (case/space-insensitive) and show all matching rows
tmp = player_name_id.copy()
tmp['NBAName_key'] = tmp['NBAName'].str.strip().str.lower()

duplicates = (
    tmp[tmp.duplicated(subset='NBAName_key', keep=False)]
    .sort_values(['NBAName_key', 'NBAID'])
    [['NBAName', 'NBAID']]
)

# For every name, check if they are in the player_teams dataset
for name in duplicates['NBAName'].unique():
    matching_players = player_name_id[player_name_id['NBAName'] == name]
    for _, row in matching_players.iterrows():
        player_id = row['NBAID']
        teams = player_teams[player_teams['Player'] == name]['Team'].unique()
        if len(teams) > 0:
            print(f"Player: {name}, NBAID: {player_id}, Teams: {', '.join(teams)}")

print(player_name_id.shape)

In [ ]:
ids_to_delete = [202419, 76993, 78190, 76616]
player_name_id = player_name_id[~player_name_id['NBAID'].isin(ids_to_delete)]

print(player_name_id.shape)

#### **Data Merging**

In [ ]:
# Start with player teams dataset
player_data = player_teams.copy()
player_data.head()

In [ ]:
# Check for nan
print(player_data.isna().sum())
print(player_data.shape)

In [ ]:
# Merge with player name id
player_data = pd.merge(player_data, player_name_id, left_on='Player', right_on='NBAName', how='left')
player_data = player_data[['NBAID', 'Player', 'Team', 'Pos']]
player_data.head()

In [ ]:
# Check for nan
print(player_data.isna().sum())
print(player_data.shape)

In [ ]:
# Merge with team id to get the team id
player_data = pd.merge(player_data, team_id, left_on='Team', right_on='BBRef_Team_Abbreviation', how='left')
player_data.head()

In [ ]:
# Check for nan
print(player_data.isna().sum())
print(player_data.shape)

In [ ]:
# Merge with height and weight
player_data = pd.merge(player_data, height_weight, left_on='Player', right_on='name', how='left')
player_data = player_data[['NBAID', 'Player', 'Team', 'Pos', 'NBA_Current_Link_ID', 'hgt', 'weight']]
player_data.head()

In [ ]:
# Manually add the height and weight for the players with missing values
player_data.loc[player_data['Player'] == 'jj obrien', 'hgt'] = 79
player_data.loc[player_data['Player'] == 'jj obrien', 'weight'] = 228

In [ ]:
# Check for nan
print(player_data.isna().sum())
print(player_data.shape)

In [ ]:
# Merge with jersey numbers
player_data = pd.merge(player_data, jersey_numbers, left_on=['Player', 'Team'], right_on=['name', 'team'], how='left')
player_data = player_data[['NBAID', 'Player', 'Team', 'Pos', 'NBA_Current_Link_ID', 'hgt', 'weight', 'jersey_number']]
player_data.head()

In [ ]:
# Check for nan
print(player_data.isna().sum())
print(player_data.shape)

In [ ]:
# merge with player stats from last season
player_data = pd.merge(player_data, players_stats_2014_15, left_on='Player', right_on='Player', how='left')
player_data.head()

In [ ]:
# Check for nan
print(player_data.isna().sum())
print(player_data.shape)

In [ ]:
# Change column names
column_renames = {
    'NBAID': 'player_id',
    'Player': 'player_name',
    'Team': 'team',
    'Pos': 'position',
    'NBA_Current_Link_ID': 'team_id',
    'hgt': 'height',
    'weight': 'weight',
    'jersey_number': 'jersey_number',
    'ORB': 'orb_per_100',
    'DRB': 'drb_per_100',
    'TRB': 'trb_per_100'
}

player_data = player_data.rename(columns=column_renames)

In [ ]:
# Reorder columns
player_data = player_data[['player_id', 'player_name', 'team_id', 'team', 'position', 'height', 'weight', 'jersey_number', 'orb_per_100', 'drb_per_100', 'trb_per_100']]
player_data.head()

In [ ]:
# Save to csv
player_data.to_csv('../../data/player/processed/player_data.csv', index=False, encoding='utf-8')